# Aula 16 - Notebook: Circuitos Eulerianos e Inspeção Autônoma de Tubulações

Neste notebook modelamos a malha física de manutenção preventiva e ensaios não-destrutivos (NDT) da **Planta Piloto de Produção de Biodiesel por Transesterificação em Batelada (Grupo 5)**.

### Objetivos Práticos:
1. Modelar a malha física bidirecional de inspeção como um grafo não-dirigido $G = (V, E)$;
2. Verificar analiticamente o Teorema de Euler calculando a paridade de graus de cada equipamento de processo;
3. Implementar o **Algoritmo de Hierholzer** com complexidade de tempo linear ótima $\mathcal{O}(|E|)$;
4. Gerar o circuito euleriano fechado com partida e retorno na doca de recarga (`Base`);
5. Analisar cenários alternativos (partida em outros nós e resolução do Problema do Carteiro Chinês).


## 1. Funções de Formatação Visual (ASCII SCADA)

Utilitários para formatação tabular de telemetria e exibição de matrizes de adjacência.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura padrao SCADA."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D numerica em tabela ASCII com alinhamento a direita."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)


## 2. Classe GrafoInspecaoEuleriano e Algoritmo de Hierholzer

Modelagem orientada a objetos com suporte a grafos não-dirigidos, armazenamento de metadados de dutos, verificação de paridade de graus e construção do circuito euleriano via pilha explícita.


In [2]:
from collections import defaultdict
from typing import List, Tuple, Dict, Any

class GrafoInspecaoEuleriano:
    """Modelagem de rede de inspecao robotica nao-dirigida com Algoritmo de Hierholzer."""
    def __init__(self):
        self.adj = defaultdict(list)
        self.pesos = {}
        
    def adicionar_duto(self, u: str, v: str, id_duto: str, comprimento_m: float = 10.0):
        """Cadastra tubulacao bidirecional de inspecao com identificador e peso metrico."""
        self.adj[u].append((v, id_duto))
        self.adj[v].append((u, id_duto))
        self.pesos[id_duto] = comprimento_m
        
    def graus_vertices(self) -> Dict[str, int]:
        """Retorna o grau de incidencia topologica de cada vertice."""
        return {v: len(viz) for v, viz in self.adj.items()}
        
    def verificar_euleriano(self) -> Tuple[bool, List[str]]:
        """Aplica o Teorema de Euler: retorna True se todos os graus forem pares."""
        impares = [v for v, viz in self.adj.items() if len(viz) % 2 != 0]
        return len(impares) == 0, impares
        
    def calcular_circuito_hierholzer(self, inicio: str) -> List[str]:
        """Executa o Algoritmo de Hierholzer com pilha explicita e consumo destrutivo de arestas."""
        adj_copia = {u: list(viz) for u, viz in self.adj.items()}
        pilha = [inicio]
        circuito = []
        while pilha:
            u = pilha[-1]
            if adj_copia[u]:
                v, id_e = adj_copia[u].pop()
                adj_copia[v].remove((u, id_e))
                pilha.append(v)
            else:
                circuito.append(pilha.pop())
        circuito.reverse()
        return circuito
        
    def calcular_distancia_circuito(self, circuito: List[str]) -> float:
        """Calcula a distancia total percorrida pelo circuito euleriano."""
        adj_copia = {u: list(viz) for u, viz in self.adj.items()}
        dist_total = 0.0
        for i in range(len(circuito) - 1):
            u, v = circuito[i], circuito[i+1]
            for j, (viz, id_e) in enumerate(adj_copia[u]):
                if viz == v:
                    dist_total += self.pesos[id_e]
                    adj_copia[u].pop(j)
                    adj_copia[v].remove((u, id_e))
                    break
        return dist_total


## 3. Modelagem e Instanciação da Malha da Planta de Biodiesel (Grupo 5)

Cadastro dos 6 nós principais (`Base`, `TK101`, `MAN101`, `R200`, `TK201`, `TK300`) e dos 9 segmentos de ensaio não-destrutivo.


In [3]:
# Instanciação do grafo e inserção dos dutos com comprimentos reais em metros
g_insp = GrafoInspecaoEuleriano()

# 1. Anel Periférico Externo (Ligação com a Base de Recarga)
g_insp.adicionar_duto("Base", "TK101", "d1", 10.0)
g_insp.adicionar_duto("TK101", "MAN101", "d2", 15.0)
g_insp.adicionar_duto("MAN101", "R200", "d3", 25.0)
g_insp.adicionar_duto("R200", "TK201", "d4", 18.0)
g_insp.adicionar_duto("TK201", "TK300", "d5", 20.0)
g_insp.adicionar_duto("TK300", "Base", "d6", 12.0)

# 2. Anel Interno de Manifolds e Linhas Secundárias de Reciclo
g_insp.adicionar_duto("TK101", "R200", "d7", 28.0)
g_insp.adicionar_duto("R200", "MAN101", "d8", 22.0)
g_insp.adicionar_duto("MAN101", "TK101", "d9", 15.0)

# Relatório tabular de graus topológicos
dados_graus = []
for no, grau in g_insp.graus_vertices().items():
    paridade = "Par" if grau % 2 == 0 else "Ímpar"
    status = "Válido" if grau % 2 == 0 else "Inválido"
    dados_graus.append({
        "Equipamento / Nó": no,
        "Grau (deg)": grau,
        "Paridade": paridade,
        "Status Euleriano": status
    })

print("=== BALANÇO DE GRAUS TOPOLÓGICOS (MALHA DE INSPEÇÃO DE BIODIESEL) ===")
print(formatar_tabela(dados_graus))


=== BALANÇO DE GRAUS TOPOLÓGICOS (MALHA DE INSPEÇÃO DE BIODIESEL) ===
Equipamento / Nó | Grau (deg) | Paridade | Status Euleriano
-----------------+------------+----------+-----------------
Base             | 2          | Par      | Válido          
TK101            | 4          | Par      | Válido          
MAN101           | 4          | Par      | Válido          
R200             | 4          | Par      | Válido          
TK201            | 2          | Par      | Válido          
TK300            | 2          | Par      | Válido          


## 4. Verificação do Teorema de Euler e Execução de Hierholzer

Cálculo do circuito euleriano com partida e retorno na doca `Base` e análise comparativa de ponto de partida alternativo (`R200`).


In [4]:
eul, imp = g_insp.verificar_euleriano()
print("1. VERIFICAÇÃO FORMAL DO TEOREMA DE EULER:")
print(f"Grafo é Euleriano: {eul} | Vértices Ímpares: {imp}\n")

# Execucao de Hierholzer a partir da Base de recarga
rota_base = g_insp.calcular_circuito_hierholzer("Base")
dist_base = g_insp.calcular_distancia_circuito(rota_base)
print("2. CIRCUITO EULERIANO DE INSPEÇÃO (Partida na Base):")
print("Rota do Robô:", " -> ".join(rota_base))
print(f"Distância Métrica Total Inspecionada: {dist_base:.1f} m")
print(f"Total de Trechos Percorridos: {len(rota_base) - 1} | Repetições de Dutos: 0\n")

# Execucao alternativa partindo do reator R200
rota_r200 = g_insp.calcular_circuito_hierholzer("R200")
dist_r200 = g_insp.calcular_distancia_circuito(rota_r200)
print("3. CIRCUITO EULERIANO ALTERNATIVO (Partida no Reator R200):")
print("Rota do Robô:", " -> ".join(rota_r200))
print(f"Distância Métrica Total Inspecionada: {dist_r200:.1f} m")

# Assercoes de validacao formal
assert eul is True
assert len(imp) == 0
assert rota_base[0] == "Base" and rota_base[-1] == "Base"
assert rota_r200[0] == "R200" and rota_r200[-1] == "R200"
assert len(rota_base) == 10
assert dist_base == 165.0
assert dist_r200 == 165.0


1. VERIFICAÇÃO FORMAL DO TEOREMA DE EULER:
Grafo é Euleriano: True | Vértices Ímpares: []

2. CIRCUITO EULERIANO DE INSPEÇÃO (Partida na Base):
Rota do Robô: Base -> TK300 -> TK201 -> R200 -> MAN101 -> TK101 -> R200 -> MAN101 -> TK101 -> Base
Distância Métrica Total Inspecionada: 165.0 m
Total de Trechos Percorridos: 9 | Repetições de Dutos: 0

3. CIRCUITO EULERIANO ALTERNATIVO (Partida no Reator R200):
Rota do Robô: R200 -> MAN101 -> TK101 -> Base -> TK300 -> TK201 -> R200 -> MAN101 -> TK101 -> R200
Distância Métrica Total Inspecionada: 165.0 m


## 5. Simulação de Cenário Não-Euleriano e Resolução do Carteiro Chinês

Demonstração do impacto da inserção de tubulação redundante que introduz vértices ímpares e reparação de paridade via duplicação ótima.


In [5]:
# Criacao de cenario com vertice impar
g_impar = GrafoInspecaoEuleriano()
for u, vizs in g_insp.adj.items():
    for v, id_e in vizs:
        if id_e not in g_impar.pesos:
            g_impar.adicionar_duto(u, v, id_e, g_insp.pesos[id_e])

# Insercao de d10 paralelo
g_impar.adicionar_duto("TK300", "TK201", "d10", 20.0)
eul_mod, imp_mod = g_impar.verificar_euleriano()
print("1. INSERÇÃO DE DUTO REDUNDANTE D10 (TK300 <-> TK201):")
print(f"Grafo é Euleriano: {eul_mod} | Vértices com Grau Ímpar: {sorted(imp_mod)}\n")

# Resolucao pelo Carteiro Chines: duplicacao de d5 para restaurar paridade
g_impar.adicionar_duto("TK201", "TK300", "d5_dup", 20.0)
eul_rep, imp_rep = g_impar.verificar_euleriano()
rota_cpp = g_impar.calcular_circuito_hierholzer("Base")
dist_cpp = g_impar.calcular_distancia_circuito(rota_cpp)

print("2. APLICAÇÃO DO PROBLEMA DO CARTEIRO CHINÊS (CPP):")
print("Emparelhamento de Menor Custo: Duplicar duto direto d5 entre TK201 e TK300 (+20.0 m)")
print(f"Novo Status Euleriano após Augmentação: {eul_rep} | Vértices Ímpares: {imp_rep}")
print("Circuito de Inspeção Aumentado:")
print(" -> ".join(rota_cpp))
print(f"Distância Aumentada Total: {dist_cpp:.1f} m\n")

# Assercoes do experimento
assert eul_mod is False
assert set(imp_mod) == {"TK201", "TK300"}
assert eul_rep is True
assert dist_cpp == 185.0
print("TODAS AS ASSERÇÕES DE GRAFOS EULERIANOS FORAM VALIDADAS COM SUCESSO!")


1. INSERÇÃO DE DUTO REDUNDANTE D10 (TK300 <-> TK201):
Grafo é Euleriano: False | Vértices com Grau Ímpar: ['TK201', 'TK300']

2. APLICAÇÃO DO PROBLEMA DO CARTEIRO CHINÊS (CPP):
Emparelhamento de Menor Custo: Duplicar duto direto d5 entre TK201 e TK300 (+20.0 m)
Novo Status Euleriano após Augmentação: True | Vértices Ímpares: []
Circuito de Inspeção Aumentado:
Base -> TK300 -> TK201 -> TK300 -> TK201 -> R200 -> MAN101 -> TK101 -> R200 -> MAN101 -> TK101 -> Base
Distância Aumentada Total: 185.0 m

TODAS AS ASSERÇÕES DE GRAFOS EULERIANOS FORAM VALIDADAS COM SUCESSO!
